# **Applied Statistic for Social Science**
<img src="../../figures/upc_logo.png" width="350"/>

## 📊 **Week 02: Data and Visualization**

**Docente**: Enzo Infantes Zúñiga  
**Contacto**: <pcefeinf@upc.edu.pe>  
**LinkedIn**: [enzo-infantes](https://www.linkedin.com/in/enzo-infantes/)

## 🎯 **Session objectives:**

By the end of the lecture, students will be able to:   

- Recognize the structure of the data.
- Organize a variable using a frequency table.
- Select the best graph to display the data based on the variable type.
- Interpret the graphs properly.

# **1. Libraries**

We will exclusively use the standard Python data analysis toolkit:

| Library | What do we use it for? |
| --- | --- |
| `pandas` | Read datasets, manipulate tables (DataFrames), and build frequency tables |
| `numpy` | Numerical operations (logarithms, ranges, rounding) |
| `matplotlib` | Basic graphing engine |
| `seaborn` | Statistical plots with less code and better aesthetics (built on top of matplotlib) |

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import os

main_path = os.path.dirname(os.path.dirname(os.getcwd()))
data_path = os.path.join(main_path, 'data', 'S02')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

'c:\\Users\\einfantesz\\Documents\\Estadistica\\Applied_Statistic'

# **2. Data**

The dataset is in **Stata (`.dta`)** format. `pandas` reads it directly using `pd.read_stata()`.

Two important decisions when loading:
- `convert_categoricals=False`: we keep the **numeric codes** (1, 2, 3...) instead of the Stata labels.
  This allows us to control what each code means (we will do this in Section 3).
- The original dataset has 73 columns; we will keep only the ones we will use in this session.

In [ ]:
# Seleccionamos las variables que usaremos y las renombramos cuando el nombre es poco claro.
# Usar un diccionario {nombre_original: nombre_nuevo} es la forma más ordenada de hacerlo.
variables = {
    # Identificación
    'aÑo'           : 'anio',          # año de la encuesta
    'conglome'      : 'conglome',
    'vivienda'      : 'vivienda',
    'hogar'         : 'hogar',
    'codperso'      : 'codperso',
    # Geografía
    'departamento'  : 'departamento',
    'region'        : 'region',
    'rural'         : 'rural',
    'estrato'       : 'estrato',
    # Características individuales
    'p207'          : 'sexo',
    'edad'          : 'edad',
    'p209'          : 'estado_civil',
    'civil'         : 'casado',
    'migrante'      : 'migrante',
    # Educación
    'p301a'         : 'nivel_educ',
    'educ'          : 'anios_educ',
    'nivel'         : 'nivel_habilidad',
    # Empleo e ingresos
    'actividad'     : 'sector',
    'ocupinf'       : 'informal',
    'whora'         : 'horas_semana',
    'ing_nom'       : 'ing_nominal',
    'ing_real'      : 'ing_real',
    'ln_ing_real'   : 'ln_ing_real',
    # Variables a nivel de departamento
    'capital_humano': 'capital_humano',
    'pbi_per_capita': 'pbi_pc',
    'gasto_educ'    : 'gasto_educ',
}

# **3. Describing the Data**

### **3.1 Dataset Structure**

Let's recall the logic of a structured dataset:

- **Row** → one observation (here: one employed person in a given year).
- **Column** → one variable.
- **Cell** → the observed value of that variable for that person.

Since ENAHO is conducted every year with a different sample of households, this dataset is a collection of **repeated cross-sections** for 2010, 2015, and 2020.

### **3.2 Data Dictionary**

Definitions are taken from the ENAHO technical specifications and the project construction logic 
(files `Capital_Humano_2020.do` and `Estimaciones_Capital_Humano.do`).

**Identification**

| Variable | Definition | Values |
|---|---|---|
| `anio` | Survey year | 2010, 2015, 2020 |
| `conglome`, `vivienda`, `hogar`, `codperso` | ENAHO identifiers: cluster, dwelling, household, and person order number | Text. The combination of all 4 uniquely identifies a person within a given year |

**Geography**

| Variable | Definition | Values |
|---|---|---|
| `departamento` | Department of residence (first 2 digits of ubigeo) | 1 Amazonas · 2 Áncash · 3 Apurímac · 4 Arequipa · 5 Ayacucho · 6 Cajamarca · 7 Callao · 8 Cusco · 9 Huancavelica · 10 Huánuco · 11 Ica · 12 Junín · 13 La Libertad · 14 Lambayeque · 15 Lima · 16 Loreto · 17 Madre de Dios · 18 Moquegua · 19 Pasco · 20 Piura · 21 Puno · 22 San Martín · 23 Tacna · 24 Tumbes · 25 Ucayali |
| `region` | Natural region aggregated from the ENAHO geographic domain | 0 Coast · 1 Metropolitan Lima · 2 Highlands (*Sierra*) · 3 Rainforest (*Selva*) |
| `rural` | Rural area according to geographic stratum (INEI: small populated centers / rural enumeration areas) | 0 Urban · 1 Rural |
| `estrato` | Geographic stratum based on population size of the center | 1: 500,000+ inhab. · 2: 100,000–499,999 · 3: 50,000–99,999 · 4: 20,000–49,999 · 5: 2,000–19,999 · 6: 500–1,999 · 7: Composite rural enumeration area (AER) · 8: Simple rural enumeration area (AER) (7 & 8 = rural) |

**Individual Characteristics**

| Variable | Definition | Values |
|---|---|---|
| `sexo` | Sex of the person (ENAHO p207) | 1 Male · 2 Female |
| `edad` | Age in completed years (p208a) | 15 to 65 (sample restricted to working age population) |
| `estado_civil` | Marital or cohabiting status (p209) | 1 Cohabiting · 2 Married · 3 Widowed · 4 Divorced · 5 Separated · 6 Single |
| `casado` | Lives with a partner? Constructed as cohabiting or married | 0 No · 1 Yes |
| `migrante` | Lived in a different department 5 years ago | 0 Non-migrant · 1 Migrant |

**Education**

| Variable | Definition | Values |
|---|---|---|
| `nivel_educ` | Highest educational level completed (p301a) | 1 No level · 2 Early childhood · 3 Incomplete primary · 4 Complete primary · 5 Incomplete secondary · 6 Complete secondary · 7 Incomplete non-university higher ed · 8 Complete non-university higher ed · 9 Incomplete university higher ed · 10 Complete university higher ed · 11 Master's / Doctorate |
| `anios_educ` | Years of education completed, converted from level and grade (complete primary = 6, complete secondary = 11, complete non-univ = 14, complete univ = 16, postgraduate = 16 + years) | 0 to 18 |
| `nivel_habilidad` | Skill level classification based on years of education | 0 Low (≤ 11 years) · 1 Medium (12 to 15 years) · 2 High (≥ 16 years) |

**Employment and Earnings**

| Variable | Definition | Values |
|---|---|---|
| `sector` | Economic sector of main occupation (ISIC Rev. 3 grouping based on p506) | 1 Agriculture · 2 Fishing · 3 Mining · 4 Manufacturing · 5 Electricity, water, and gas · 6 Construction · 7 Commerce · 8 Business services (transport, finance, real estate, professional) · 9 Other services (public admin, education, health, social work) |
| `informal` | Informal employment status of main occupation (INEI) | 1 Informal employment · 2 Formal employment |
| `horas_semana` | Hours worked per week across all jobs | Positive values (max 98) |
| `ing_nominal` | Sum of reported labor income (dependent + independent + secondary + extraordinary), in nominal soles | > 0. **Note:** mixes amounts of different periodicity (monthly and annual); hence `ing_real` is preferred |
| `ing_real` | **Annualized and deflated** labor income by INEI (comparable across years and regions), in soles | > 0 |
| `ln_ing_real` | Natural logarithm of `ing_real` | ≈ 2.5 to 13.7 |

**Department-level Variables** (same value for all individuals in a given department–year)

| Variable | Definition | Values |
|---|---|---|
| `capital_humano` | % of the department's labor force with completed higher education | 0 to 100 |
| `pbi_pc` | Department GDP per capita, at constant 2007 prices | Thousands of soles |
| `gasto_educ` | Department public expenditure on education | Millions of soles |

### **3.3 What type of variable is each one?**

Classifying the variable **before** plotting determines which graph to use (S02, *Types of Charts*).

| Type | Subtype | Dataset Variables | Natural Graph |
|---|---|---|---|
| Qualitative | Nominal (no order) | `sexo`, `departamento`, `region`, `rural`, `sector`, `informal`, `migrante`, `estado_civil` | Bar / column charts (pie charts only for few categories) |
| Qualitative | Ordinal (ordered) | `nivel_educ`, `nivel_habilidad`, `estrato` | Bar charts respecting order; allows cumulative frequency |
| Quantitative | Discrete (integer values) | `edad`, `anios_educ`, `horas_semana` | Histogram / bar charts by value |
| Quantitative | Continuous (any value in a range) | `ing_real`, `ing_nominal`, `ln_ing_real`, `capital_humano`, `pbi_pc`, `gasto_educ` | Histogram (grouped data), scatter plot between two variables |

> Note: storing a variable as a number (1, 2, 3) does **not** make it quantitative.
> `sector = 4` is not "double" `sector = 2`; it is merely a code.

In [ ]:
# Diccionarios con las etiquetas de cada código. Los usaremos para que tablas y gráficos muestren texto y no números.
etq_sexo   = {1: 'Hombre', 2: 'Mujer'}
etq_region = {0: 'Costa', 1: 'Lima Metropolitana', 2: 'Sierra', 3: 'Selva'}
etq_rural  = {0: 'Urbano', 1: 'Rural'}
etq_nivel  = {0: 'Baja educación', 1: 'Educación media', 2: 'Alta educación'}
etq_inform = {1: 'Informal', 2: 'Formal'}
etq_sector = {1: 'Agricultura', 2: 'Pesca', 3: 'Minería', 4: 'Manufactura', 5: 'Electricidad, agua y gas',
              6: 'Construcción', 7: 'Comercio', 8: 'Servicios', 9: 'Otros servicios'}
etq_educ   = {1: 'Sin nivel', 2: 'Inicial', 3: 'Primaria incompleta', 4: 'Primaria completa',
              5: 'Secundaria incompleta', 6: 'Secundaria completa', 7: 'Sup. no univ. incompleta',
              8: 'Sup. no univ. completa', 9: 'Sup. univ. incompleta', 10: 'Sup. univ. completa',
              11: 'Maestría / doctorado'}
etq_dpto   = {1: 'Amazonas', 2: 'Áncash', 3: 'Apurímac', 4: 'Arequipa', 5: 'Ayacucho', 6: 'Cajamarca',
              7: 'Callao', 8: 'Cusco', 9: 'Huancavelica', 10: 'Huánuco', 11: 'Ica', 12: 'Junín',
              13: 'La Libertad', 14: 'Lambayeque', 15: 'Lima', 16: 'Loreto', 17: 'Madre de Dios',
              18: 'Moquegua', 19: 'Pasco', 20: 'Piura', 21: 'Puno', 22: 'San Martín', 23: 'Tacna',
              24: 'Tumbes', 25: 'Ucayali'}

# .map(diccionario) reemplaza cada código por su etiqueta. Creamos columnas nuevas con sufijo _lbl
# para conservar también el código original.
df['sexo_lbl']    = df['sexo'].map(etq_sexo)
df['region_lbl']  = df['region'].map(etq_region)
df['rural_lbl']   = df['rural'].map(etq_rural)
df['nivel_lbl']   = df['nivel_habilidad'].map(etq_nivel)
df['informal_lbl']= df['informal'].map(etq_inform)
df['sector_lbl']  = df['sector'].map(etq_sector)
df['educ_lbl']    = df['nivel_educ'].map(etq_educ)
df['dpto_lbl']    = df['departamento'].map(etq_dpto)

# Verificamos con una muestra de columnas
df[['anio', 'sexo', 'sexo_lbl', 'sector', 'sector_lbl', 'nivel_habilidad', 'nivel_lbl']].head()

### **3.4 Missing values**

Before describing the data, we need to know **how many missing values exist** and in which variables.
In this dataset, filters have already been applied in Stata (removing individuals with missing income or hours), so we expect few missing observations.

### **3.5 Frequency table for a qualitative variable**

Session notation:

- $f_i$ = absolute frequency → **how many?**
- $h_i = f_i / n$ = relative frequency → **what proportion?**
- $100 \, h_i$ = percentage
- $F_i = \sum_{j \le i} f_j$ = cumulative frequency → **only makes sense if the variable has a natural order**

We build a small helper function so we don't repeat code.

### **3.7 Quick Numerical Summary**

`describe()` provides count, mean, standard deviation, minimum, quartiles, and maximum.
Measures of central tendency and dispersion will be studied in detail in weeks 3 and 4; here, we use them only to get an initial idea of the **range** of each variable.